In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

os.makedirs('../models/health_score', exist_ok=True)

print("✅ Libraries loaded")
print(f"   XGBoost version: {xgb.__version__}")

✅ Libraries loaded
   XGBoost version: 3.2.0


In [2]:
print("Loading health score features...")
df = pd.read_csv('../data/processed/health_score_features.csv')
print(f"✅ Loaded: {df.shape}")
print(f"\nHealth Score Stats:")
print(df['health_score'].describe().round(2))

Loading health score features...
✅ Loaded: (472950, 17)

Health Score Stats:
count    472950.00
mean         93.60
std          10.01
min          18.60
25%          90.00
50%         100.00
75%         100.00
max         100.00
Name: health_score, dtype: float64


In [3]:
feature_cols = [
    'age', 'bmi',
    'avg_systolic', 'avg_diastolic', 'bp_variability',
    'avg_glucose', 'sugar_variability',
    'avg_pulse', 'adherence_rate', 'missed_doses',
    'has_chronic_condition', 'allergies_count',
    'physical_activity_score', 'diet_quality_score',
    'stress_score', 'sleep_score'
]

X = df[feature_cols]
y = df['health_score']

print(f"✅ Features shape: {X.shape}")
print(f"✅ Target range: {y.min():.1f} - {y.max():.1f}")
print(f"✅ Target mean:  {y.mean():.1f}")

✅ Features shape: (472950, 16)
✅ Target range: 18.6 - 100.0
✅ Target mean:  93.6


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train set: {X_train_scaled.shape}")
print(f"✅ Test set:  {X_test_scaled.shape}")
print(f"\nTrain target stats:")
print(f"   Mean:  {y_train.mean():.1f}")
print(f"   Std:   {y_train.std():.1f}")
print(f"   Min:   {y_train.min():.1f}")
print(f"   Max:   {y_train.max():.1f}")

✅ Train set: (378360, 16)
✅ Test set:  (94590, 16)

Train target stats:
   Mean:  93.6
   Std:   10.0
   Min:   18.6
   Max:   100.0


In [5]:
print("Training XGBoost health score regressor...")
print("This will take 3-5 minutes due to 472k rows...")

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50
)

print("\n✅ Model trained successfully!")

Training XGBoost health score regressor...
This will take 3-5 minutes due to 472k rows...
[0]	validation_0-rmse:9.30890
[50]	validation_0-rmse:2.21267
[100]	validation_0-rmse:1.93001
[150]	validation_0-rmse:1.84025
[200]	validation_0-rmse:1.79483
[250]	validation_0-rmse:1.76599
[299]	validation_0-rmse:1.74587

✅ Model trained successfully!


In [6]:
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"📊 Model Performance:")
print(f"   MAE:   {mae:.2f} points")
print(f"   RMSE:  {rmse:.2f} points")
print(f"   R²:    {r2:.4f}")
print(f"\nSample Predictions vs Actual:")
sample_idx = np.random.choice(len(y_test), 10, replace=False)
y_test_arr = np.array(y_test)
for i in sample_idx:
    print(f"   Actual: {y_test_arr[i]:.1f}  →  Predicted: {y_pred[i]:.1f}  (diff: {abs(y_test_arr[i]-y_pred[i]):.1f})")

📊 Model Performance:
   MAE:   1.21 points
   RMSE:  1.75 points
   R²:    0.9700

Sample Predictions vs Actual:
   Actual: 83.2  →  Predicted: 84.3  (diff: 1.1)
   Actual: 62.8  →  Predicted: 68.6  (diff: 5.8)
   Actual: 92.4  →  Predicted: 92.1  (diff: 0.3)
   Actual: 75.0  →  Predicted: 73.4  (diff: 1.6)
   Actual: 100.0  →  Predicted: 99.2  (diff: 0.8)
   Actual: 100.0  →  Predicted: 98.4  (diff: 1.6)
   Actual: 97.2  →  Predicted: 98.8  (diff: 1.6)
   Actual: 96.3  →  Predicted: 98.0  (diff: 1.7)
   Actual: 100.0  →  Predicted: 100.5  (diff: 0.5)
   Actual: 100.0  →  Predicted: 98.3  (diff: 1.7)


In [7]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("📊 Feature Importance (Health Score):")
for _, row in importance_df.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"   {row['feature']:<30} {bar} {row['importance']:.4f}")

📊 Feature Importance (Health Score):
   avg_glucose                    ████████████████████████████████████████ 0.4018
   bmi                            ███████████ 0.1167
   physical_activity_score        ██████████ 0.1023
   avg_systolic                   ████████ 0.0872
   adherence_rate                 ██████ 0.0605
   has_chronic_condition          █████ 0.0533
   stress_score                   ███ 0.0367
   sugar_variability              ███ 0.0347
   missed_doses                   ██ 0.0284
   avg_diastolic                  ██ 0.0233
   diet_quality_score             ██ 0.0206
   sleep_score                    █ 0.0199
   bp_variability                 █ 0.0124
   age                             0.0018
   avg_pulse                       0.0003
   allergies_count                 0.0002


In [8]:
joblib.dump(model, '../models/health_score/health_score_model.pkl')
joblib.dump(scaler, '../models/health_score/health_score_scaler.pkl')

metadata = {
    "features": feature_cols,
    "target": "health_score",
    "type": "regression",
    "score_range": "10-100",
    "n_samples_train": int(X_train_scaled.shape[0]),
    "n_samples_test": int(X_test_scaled.shape[0]),
    "mae": round(float(mae), 4),
    "rmse": round(float(rmse), 4),
    "r2_score": round(float(r2), 4),
    "model_version": "1.0",
    "trained_on": pd.Timestamp.now().strftime("%Y-%m-%d")
}

with open('../models/health_score/health_score_features.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ All files saved to models/health_score/:")
print("   health_score_model.pkl")
print("   health_score_scaler.pkl")
print("   health_score_features.json")
print(f"\n📊 Final Summary:")
print(f"   MAE:   {mae:.2f} points")
print(f"   RMSE:  {rmse:.2f} points")
print(f"   R²:    {r2:.4f}")

✅ All files saved to models/health_score/:
   health_score_model.pkl
   health_score_scaler.pkl
   health_score_features.json

📊 Final Summary:
   MAE:   1.21 points
   RMSE:  1.75 points
   R²:    0.9700
